# Kaggle Notebook Pipeline

**Run on [kaggle.com](https://www.kaggle.com) only** — not on your laptop.

1. **Settings → Internet → On** (needed for `git clone` + `pip install`)
2. **Settings → Accelerator → None** (CPU is enough for Phase 1)
3. **Add Data** → attach input dataset(s) listed in the config cell below
4. Run all cells, then **Save Version** → **Save as Dataset** to pass `data/` to the next notebook

**Inputs:** `PIPELINE_INPUT` = output from notebook 02 (epoch `.npy`).


In [ ]:
# --- Kaggle configuration (edit slugs to match your input datasets) ---
REPO_URL = "https://github.com/RandomPerson5571/ad_eeg.git"
REPO_BRANCH = "main"
PROJECT_DIR = "/kaggle/working/ad_eeg"

# Kaggle dataset slug with raw EEG (must contain EEG_data/dataset2/ and dataset3/)
RAW_EEG_INPUT = "REPLACE_WITH_RAW_EEG_DATASET_SLUG"

# Optional: output from a prior pipeline notebook (must contain data/ at root)
PIPELINE_INPUT = None  # e.g. "REPLACE_WITH_PRIOR_PIPELINE_OUTPUT_SLUG"


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError(
        "This notebook runs on Kaggle only. "
        "Upload to kaggle.com, enable Internet, attach input datasets, then run."
    )

PROJECT_DIR = Path(PROJECT_DIR)


def run(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


if not PROJECT_DIR.exists():
    run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
run(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=PROJECT_DIR)
print(f"Project root: {PROJECT_DIR.resolve()}", flush=True)


def _find_eeg_root(slug: str) -> Path | None:
    base = Path("/kaggle/input") / slug
    if not base.exists():
        return None
    if (base / "EEG_data").is_dir():
        return base / "EEG_data"
    if (base / "dataset2").is_dir():
        return base
    for child in base.iterdir():
        if child.is_dir() and (child / "EEG_data").is_dir():
            return child / "EEG_data"
        if child.is_dir() and (child / "dataset2").is_dir():
            return child
    return None


eeg_link = PROJECT_DIR / "EEG_data"
if RAW_EEG_INPUT:
    src = _find_eeg_root(RAW_EEG_INPUT)
    if src is None:
        raise FileNotFoundError(
            f"Raw EEG not found for slug '{RAW_EEG_INPUT}'. "
            "Add Data → your dataset with EEG_data/dataset2/ and dataset3/."
        )
    if eeg_link.is_symlink():
        eeg_link.unlink()
    elif eeg_link.is_dir() and not eeg_link.is_symlink():
        pass
    elif eeg_link.exists():
        eeg_link.unlink()
    if not eeg_link.exists():
        os.symlink(src, eeg_link)
    print(f"EEG_data → {src}", flush=True)

if PIPELINE_INPUT:
    pipeline_src = Path("/kaggle/input") / PIPELINE_INPUT / "data"
    if not pipeline_src.exists():
        pipeline_src = Path("/kaggle/input") / PIPELINE_INPUT
        if not (pipeline_src / "preprocessed").exists() and not (pipeline_src / "audit").exists():
            raise FileNotFoundError(
                f"Pipeline input '{PIPELINE_INPUT}' has no data/ folder. "
                "Save the previous notebook version as a Dataset first."
            )
    dest = PROJECT_DIR / "data"
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(pipeline_src, dest, dirs_exist_ok=True)
    print(f"Restored pipeline data from {pipeline_src}", flush=True)


# 06 — Deep Learning (stub)

**Phase 1: no DL training code.**

## Input contract
- `data/preprocessed/{dataset}/{experiment}/epochs/sub-XXX.npy`
- Shape: `(n_epochs, n_channels, n_samples)` float32

## Planned models (backlog)
- EEGNet, ChronoNet, DeepConvNet, ShallowConvNet, Transformer


In [ ]:
from eeg.config import load_experiment, resolve_dataset
from eeg.repro import init_repro, snapshot_environment

EXPERIMENT = "baseline"
dataset_spec = resolve_dataset("eyesclosed")[0]
config = load_experiment(EXPERIMENT)

CONFIG = {
    "dataset": dataset_spec.name,
    "experiment": EXPERIMENT,
    "seed": config.get("training", {}).get("random_state", 42),
    "cv_folds": config.get("training", {}).get("cv_folds", 5),
    "feature_set": "full",
    "normalization": "zscore",
}
repro = init_repro(CONFIG["seed"])
env = snapshot_environment()
print(CONFIG)


In [ ]:
from eeg.paths import epochs_npy_dir
print("Epoch npy dir:", epochs_npy_dir(CONFIG["dataset"], CONFIG["experiment"]))
